<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Gradient Boosting & XGBoost</h1></center>

Gradient Boosting builds trees **sequentially** — each tree corrects the errors of the previous ones. XGBoost is a fast, regularised implementation.

### Topics:
1. [Theory — Sequential Boosting](#1)
2. [Dataset & Visualization](#2)
3. [Learning Rate vs n_estimators](#3)
4. [XGBoost with Early Stopping](#4)
5. [Evaluation of Model](#5)

In [ ]:
import seaborn as sns
gb_colors = ['#370617', '#9d0208', '#e85d04', '#f48c06', '#faa307']
print("Gradient Boosting Colors:")
sns.palplot(sns.color_palette(gb_colors))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, log_loss)
from sklearn.inspection import permutation_importance
try:
    import xgboost as xgb
    XGB = True
except ImportError:
    XGB = False
    print("XGBoost not installed — run: pip install xgboost")
import warnings; warnings.filterwarnings("ignore")
%matplotlib inline
np.random.seed(42)
print(f"Libraries loaded. XGBoost available: {XGB}")

## Let's Start!

<a id = "1"></a>
<center><h1 style = "background:#000000 ;color:white;border:0;font-weight:bold">Theory — Sequential Boosting</h1></center>

## How Gradient Boosting Works

Gradient Boosting fits trees **one by one**, where each new tree $h_m$ is fit to the **negative gradient** (pseudo-residuals) of the loss:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

- $\eta$ (learning rate / shrinkage) scales each tree's contribution
- Lower $\eta$ + more trees → better generalisation (but slower)

## XGBoost Extras

| Feature | Benefit |
|---|---|
| Histogram-based splits | 10-100× faster than brute-force |
| L1 + L2 regularisation | Controls leaf weights |
| Subsampling rows/cols | Adds stochasticity, reduces overfit |
| Early stopping | Auto-selects optimal n_estimators |
| Native missing handling | No imputation needed |

<SCRIPT SRC='https://cdn.mathjax.org/mathjax/latest/MathJax.js?config=TeX-AMS-MML_HTMLorMML'></SCRIPT>
<SCRIPT>MathJax.Hub.Config({ tex2jax: {inlineMath: [['$','$'],['\\(','\\)']]}});</SCRIPT>

<a id = "2"></a>
<center><h1 style = "background:#370617 ;color:white;border:0;font-weight:bold">Information About Dataset</h1></center>

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
df = X.copy(); df["target"] = y
print(f"Shape: {df.shape}")
print(df["target"].value_counts().rename({0:"malignant",1:"benign"}))
df.describe().T.head(8)

<center><h1 style = "background:#9d0208 ;color:white;border:0;font-weight:bold">Data Visualization</h1></center>

In [ ]:
top_features = ["mean radius", "mean texture", "mean area", "mean concavity"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=100)

for ax, feat in zip(axes.ravel(), top_features):
    for label, color in [(0, "#370617"), (1, "#f48c06")]:
        sns.kdeplot(df[df["target"]==label][feat], ax=ax,
                    label=data.target_names[label], color=color, fill=True, alpha=0.4)
    ax.set_title(feat); ax.legend()

plt.suptitle("Feature Distributions by Class", fontweight="bold")
plt.tight_layout(); plt.show()

<center><h1 style = "background:#e85d04 ;color:white;border:0;font-weight:bold">Train-Test Split</h1></center>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

<a id = "3"></a>
<center><h1 style = "background:#f48c06 ;color:black;border:0;font-weight:bold">Learning Rate vs n_estimators</h1></center>

In [ ]:
configs = [
    {"learning_rate": 0.3,  "n_estimators": 100,  "label": "η=0.3, 100 trees",  "color": "#370617"},
    {"learning_rate": 0.1,  "n_estimators": 200,  "label": "η=0.1, 200 trees",  "color": "#9d0208"},
    {"learning_rate": 0.05, "n_estimators": 400,  "label": "η=0.05, 400 trees", "color": "#faa307"},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

for cfg in configs:
    gb = GradientBoostingClassifier(learning_rate=cfg["learning_rate"],
                                    n_estimators=cfg["n_estimators"],
                                    max_depth=3, random_state=42)
    gb.fit(X_train, y_train)
    tr_loss = [log_loss(y_train, p) for p in gb.staged_predict_proba(X_train)]
    te_loss = [log_loss(y_test,  p) for p in gb.staged_predict_proba(X_test)]
    iters = range(1, cfg["n_estimators"]+1)
    axes[0].plot(iters, tr_loss, color=cfg["color"], lw=1.5, label=cfg["label"])
    axes[1].plot(iters, te_loss, color=cfg["color"], lw=1.5, ls="--", label=cfg["label"])

for ax, title in zip(axes, ["Train Log-Loss", "Test Log-Loss"]):
    ax.set_xlabel("Boosting round"); ax.set_ylabel("Log-Loss")
    ax.set_title(title, fontweight="bold"); ax.legend(fontsize=8)

plt.tight_layout(); plt.show()

<a id = "4"></a>
<center><h1 style = "background:#9d0208 ;color:white;border:0;font-weight:bold">XGBoost with Early Stopping</h1></center>

In [ ]:
if XGB:
    model_xgb = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric="logloss", early_stopping_rounds=30,
        random_state=42, n_jobs=-1, verbosity=0
    )
    model_xgb.fit(X_tr2, y_tr2,
                  eval_set=[(X_tr2, y_tr2), (X_val, y_val)],
                  verbose=False)

    res = model_xgb.evals_result()
    tr_ll = res["validation_0"]["logloss"]
    va_ll = res["validation_1"]["logloss"]

    plt.figure(figsize=(10, 5), dpi=100)
    plt.plot(tr_ll, color="#370617", label="Train log-loss")
    plt.plot(va_ll, color="#faa307", label="Val log-loss")
    plt.axvline(model_xgb.best_iteration, color="gray", ls="--",
                label=f"Best iter={model_xgb.best_iteration}")
    plt.xlabel("Boosting round"); plt.ylabel("Log-loss")
    plt.title("XGBoost Early Stopping", fontweight="bold")
    plt.legend(); plt.show()
    print(f"Best iteration: {model_xgb.best_iteration}")
else:
    print("XGBoost not available.")

<a id = "5"></a>
<center><h1 style = "background:#faa307 ;color:black;border:0;font-weight:bold">Evaluation of Model</h1></center>

In [ ]:
best_gb = GradientBoostingClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=3,
    subsample=0.8, random_state=42)
best_gb.fit(X_train, y_train)
pred = best_gb.predict(X_test)
prob = best_gb.predict_proba(X_test)[:, 1]

from sklearn.metrics import precision_score, recall_score, f1_score
results = [accuracy_score(y_test, pred), precision_score(y_test, pred),
           recall_score(y_test, pred), f1_score(y_test, pred),
           roc_auc_score(y_test, prob), log_loss(y_test, prob)]
pd.DataFrame({"Metric": ["Accuracy","Precision","Recall","F1","ROC-AUC","Log-Loss"],
              "Score": results}).round(4)

In [ ]:
print(classification_report(y_test, pred, target_names=data.target_names))

cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5, 4), dpi=100)
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("Confusion Matrix — Gradient Boosting")
plt.show()